In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
# Install required packages first:
# pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers pypdf docx2txt

from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain_community.llms import Ollama
import os

# =========================
# 1. LOAD FILE (AUTO DETECT)
# =========================
file_path = "C:/Users/Dell/Downloads/Nature.pdf"   # change to your file (.txt / .pdf / .docx)

if file_path.endswith(".txt"):
    loader = TextLoader(file_path)

elif file_path.endswith(".pdf"):
    loader = PyPDFLoader(file_path)

elif file_path.endswith(".docx"):
    loader = Docx2txtLoader(file_path)

else:
    raise ValueError("Unsupported file type")

documents = loader.load()

# =========================
# 2. SPLIT TEXT
# =========================
splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
docs = splitter.split_documents(documents)

# =========================
# 3. EMBEDDINGS (FIXED VERSION)
# =========================
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# =========================
# 4. VECTOR DATABASE
# =========================
db = FAISS.from_documents(docs, embeddings)

# =========================
# 5. LOAD LLM (OLLAMA)
# =========================
#  # make sure: ollama run llama3
#llm = Ollama(model="phi")
llm = Ollama(model="tinyllama")
# =========================
# 6. QUESTION LOOP
# =========================
print("\n✅ System Ready! Ask your questions (type 'exit' to quit)\n")

while True:
    query = input("👉 Question: ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    # =========================
    # 7. RETRIEVE CONTEXT (RAG)
    # =========================
    retrieved_docs = db.similarity_search(query)
    context = " ".join([doc.page_content for doc in retrieved_docs])

    # =========================
    # 8. PROMPT
    # =========================
    prompt = f"""
    You are an AI assistant.

    Answer ONLY using the context below.
    If the answer is not in the context, say "I don't know".

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # =========================
    # 9. GENERATE RESPONSE
    # =========================
    response = llm.invoke(prompt)

    print("\n💡 Answer:", response)


✅ System Ready! Ask your questions (type 'exit' to quit)



👉 Question:  Tell me about nature



💡 Answer: Yes, I can provide an answer using the given context. Nature encompases all living organisms, ecosystems, and physical features of the Earth created by humans. It provides oxygen, food, water, climate regulation, and significant physical and mental health benefits. However, human activity threatens these systems and affects the balance required for a sustainable planet.


👉 Question:  exit


Goodbye!
